# CKD NB1 — Data Audit and Target Feasibility

## Project
Trustworthy Agentic AI for Healthcare

## Objective
Audit the available NHANES 2017–March 2020 Pre-Pandemic data to determine whether a defensible laboratory-derived chronic kidney disease (CKD) research indicator can be constructed.

## Research tasks
1. Identify the available kidney-related data files and variables.
2. Inspect variable definitions, units, missingness, and participant identifiers.
3. Assess the feasibility of defining a laboratory-derived CKD research indicator.
4. Identify potential target leakage and unsuitable predictors.
5. Document the proposed target definition and limitations before modelling.

## Scope and safeguards
- This notebook is exploratory and does not train a prediction model.
- The existing diabetes model and its artifacts will remain unchanged.
- Participant-level data will not be uploaded to GitHub.
- A single NHANES examination cannot establish the persistence of kidney abnormalities required for a clinical CKD diagnosis.

## Expected output
A documented data audit and a justified decision on whether the available variables support the proposed CKD research task.

In [2]:
from pathlib import Path
import os
import sys

print("Python version:", sys.version.split()[0])
print("Current working directory:", os.getcwd())
print("\nFiles in the current directory:")

for path in sorted(Path.cwd().iterdir()):
    print(path.name + ("/" if path.is_dir() else ""))

print("\nGoogle Drive mounted:", Path("/content/drive/MyDrive").exists())

Python version: 3.13.16
Current working directory: /content

Files in the current directory:
.config/
sample_data/

Google Drive mounted: False


In [3]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [4]:
from pathlib import Path
import subprocess

REPO_URL = (
    "https://github.com/MeghaKA/"
    "Trustworthy-Agentic-AI-for-Healthcare.git"
)

PROJECT_ROOT = Path("/content/Trustworthy-Agentic-AI-for-Healthcare")

# Download the repository into Colab if it isn't already present.
if not PROJECT_ROOT.exists():
    result = subprocess.run(
        ["git", "clone", "--depth", "1", REPO_URL, str(PROJECT_ROOT)],
        capture_output=True,
        text=True
    )

    if result.returncode != 0:
        raise RuntimeError(result.stderr)

print("Repository directory exists:", PROJECT_ROOT.exists())

RAW_DATA = PROJECT_ROOT / "data" / "raw"

print("Raw data directory exists:", RAW_DATA.exists())

# Locate the NHANES XPT files.
if RAW_DATA.exists():
    xpt_files = sorted(RAW_DATA.glob("*.xpt"))

    print(f"\nNumber of XPT files found: {len(xpt_files)}")

    for path in xpt_files:
        print(path.name, f"({path.stat().st_size / 1024:.1f} KB)")
else:
    print("Could not find data/raw. Check the repository structure.")

Repository directory exists: True
Raw data directory exists: True

Number of XPT files found: 21
P_ALB_CR.xpt (816.0 KB)
P_ALQ.xpt (702.5 KB)
P_BIOPRO.xpt (3340.5 KB)
P_BMX.xpt (2461.6 KB)
P_BPXO.xpt (1015.5 KB)
P_CBC.xpt (2370.9 KB)
P_DBQ.xpt (5598.9 KB)
P_DEMO.xpt (3530.0 KB)
P_DIQ.xpt (3282.7 KB)
P_GHB.xpt (163.7 KB)
P_GLU.xpt (160.3 KB)
P_HDL.xpt (287.1 KB)
P_HIQ.xpt (1704.5 KB)
P_HSCRP.xpt (324.0 KB)
P_HUQ.xpt (852.7 KB)
P_PAQ.xpt (1290.5 KB)
P_RXQ_RX.xpt (15775.4 KB)
P_SLQ.xpt (759.0 KB)
P_SMQ.xpt (1395.1 KB)
P_TCHOL.xpt (287.1 KB)
P_TRIGLY.xpt (399.8 KB)


In [5]:
import pandas as pd
from pathlib import Path

RAW_DATA = PROJECT_ROOT / "data" / "raw"

# Kidney-related NHANES source files
kidney_files = [
    "P_ALB_CR.xpt",  # Urine albumin and creatinine
    "P_BIOPRO.xpt",  # Blood biochemistry, including kidney-related measures
    "P_DEMO.xpt",    # Demographic information and participant ID
]

for filename in kidney_files:
    file_path = RAW_DATA / filename

    print("\n" + "=" * 70)
    print("FILE:", filename)

    if not file_path.exists():
        print("File not found.")
        continue

    df = pd.read_sas(file_path, format="xport")

    print("Shape:", df.shape)
    print("\nAvailable columns:")
    print(df.columns.tolist())

    print("\nFirst 3 rows:")
    display(df.head(3))

    print("\nMissingness in each column (%):")
    missing = (df.isna().mean() * 100).round(2)
    display(missing.sort_values(ascending=False).to_frame("missing_percent"))


FILE: P_ALB_CR.xpt
Shape: (13027, 8)

Available columns:
['SEQN', 'URXUMA', 'URXUMS', 'URDUMALC', 'URXUCR', 'URXCRS', 'URDUCRLC', 'URDACT']

First 3 rows:


,SEQN,URXUMA,URXUMS,URDUMALC,URXUCR,URXCRS,URDUCRLC,URDACT
0,109264.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,109266.0,5.5,5.5,5.397605e-79,36.0,3182.4,5.397605e-79,15.28
2,109270.0,4.0,4.0,5.397605e-79,165.0,14586.0,5.397605e-79,2.42



Missingness in each column (%):


,missing_percent
URDACT,3.98
URDUCRLC,3.98
URXCRS,3.98
URXUCR,3.98
URXUMS,3.97
URDUMALC,3.97
URXUMA,3.97
SEQN,0.00



FILE: P_BIOPRO.xpt
Shape: (10409, 41)

Available columns:
['SEQN', 'LBXSATSI', 'LBDSATLC', 'LBXSAL', 'LBDSALSI', 'LBXSAPSI', 'LBXSASSI', 'LBXSC3SI', 'LBXSBU', 'LBDSBUSI', 'LBXSCLSI', 'LBXSCK', 'LBXSCR', 'LBDSCRSI', 'LBXSGB', 'LBDSGBSI', 'LBXSGL', 'LBDSGLSI', 'LBXSGTSI', 'LBDSGTLC', 'LBXSIR', 'LBDSIRSI', 'LBXSLDSI', 'LBXSOSSI', 'LBXSPH', 'LBDSPHSI', 'LBXSKSI', 'LBXSNASI', 'LBXSTB', 'LBDSTBSI', 'LBDSTBLC', 'LBXSCA', 'LBDSCASI', 'LBXSCH', 'LBDSCHSI', 'LBXSTP', 'LBDSTPSI', 'LBXSTR', 'LBDSTRSI', 'LBXSUA', 'LBDSUASI']

First 3 rows:


...

,missing_percent
LBXSLDSI,11.01
LBXSASSI,9.36
LBXSKSI,9.06
LBXSCK,9.03
LBXSBU,8.99
LBDSGBSI,8.99
LBXSATSI,8.99
LBXSC3SI,8.99
LBDSATLC,8.99
LBXSUA,8.99



FILE: P_DEMO.xpt
Shape: (15560, 29)

Available columns:
['SEQN', 'SDDSRVYR', 'RIDSTATR', 'RIAGENDR', 'RIDAGEYR', 'RIDAGEMN', 'RIDRETH1', 'RIDRETH3', 'RIDEXMON', 'DMDBORN4', 'DMDYRUSZ', 'DMDEDUC2', 'DMDMARTZ', 'RIDEXPRG', 'SIALANG', 'SIAPROXY', 'SIAINTRP', 'FIALANG', 'FIAPROXY', 'FIAINTRP', 'MIALANG', 'MIAPROXY', 'MIAINTRP', 'AIALANGA', 'WTINTPRP', 'WTMECPRP', 'SDMVPSU', 'SDMVSTRA', 'INDFMPIR']

First 3 rows:


,SEQN,SDDSRVYR,RIDSTATR,RIAGENDR,RIDAGEYR,RIDAGEMN,RIDRETH1,RIDRETH3,RIDEXMON,DMDBORN4,...,FIAINTRP,MIALANG,MIAPROXY,MIAINTRP,AIALANGA,WTINTPRP,WTMECPRP,SDMVPSU,SDMVSTRA,INDFMPIR
0,109263.0,66.0,2.0,1.0,2.0,NaN,5.0,6.0,2.0,1.0,...,2.0,NaN,NaN,NaN,NaN,7891.762435,8951.815567,3.0,156.0,4.66
1,109264.0,66.0,2.0,2.0,13.0,NaN,1.0,1.0,2.0,1.0,...,2.0,1.0,2.0,2.0,1.0,11689.747264,12271.157043,1.0,155.0,0.83
2,109265.0,66.0,2.0,1.0,2.0,NaN,3.0,3.0,2.0,1.0,...,2.0,NaN,NaN,NaN,NaN,16273.825939,16658.764203,1.0,157.0,3.06



Missingness in each column (%):


,missing_percent
RIDAGEMN,93.66
RIDEXPRG,87.96
DMDYRUSZ,80.54
AIALANGA,47.15
DMDEDUC2,40.67
DMDMARTZ,40.67
MIAPROXY,29.31
MIALANG,29.31
MIAINTRP,29.31
INDFMPIR,14.15


In [6]:
import numpy as np
import pandas as pd

# Load only the variables needed for this preliminary check.
demo = pd.read_sas(
    RAW_DATA / "P_DEMO.xpt", format="xport"
)[["SEQN", "RIDAGEYR", "RIAGENDR"]]

biopro = pd.read_sas(
    RAW_DATA / "P_BIOPRO.xpt", format="xport"
)[["SEQN", "LBXSCR", "LBXSBU"]]

urine = pd.read_sas(
    RAW_DATA / "P_ALB_CR.xpt", format="xport"
)[["SEQN", "URXUMA", "URXUCR", "URDACT"]]

# Join using the NHANES participant identifier.
audit = (
    demo.merge(biopro, on="SEQN", how="left", validate="one_to_one")
        .merge(urine, on="SEQN", how="left", validate="one_to_one")
)

print("Merged participant records:", len(audit))
print("Unique participant IDs:", audit["SEQN"].nunique())

# Inspect tiny values that may encode SAS special missing values.
measurement_cols = [
    "LBXSCR", "LBXSBU", "URXUMA", "URXUCR", "URDACT"
]

print("\nValues extremely close to zero:")
for col in measurement_cols:
    s = audit[col]
    tiny_count = (s.notna() & (s.abs() < 1e-10)).sum()
    print(f"{col}: {tiny_count} of {s.notna().sum()} non-NaN values")

print("\nMeasurement summaries before any cleaning:")
display(
    audit[measurement_cols]
    .describe(percentiles=[0.01, 0.05, 0.50, 0.95, 0.99])
    .T
)

# Adult cohort check; this is exploratory, not the final modelling cohort.
adults = audit.loc[audit["RIDAGEYR"] >= 18].copy()

print("\nParticipants aged 18 or older:", len(adults))
print("\nAdult age range:")
print(adults["RIDAGEYR"].min(), "to", adults["RIDAGEYR"].max())

print("\nAdult missingness (%):")
display((adults[measurement_cols].isna().mean() * 100).round(2))

print("\nFirst 10 rows of relevant measurements:")
display(
    adults[
        ["SEQN", "RIDAGEYR", "RIAGENDR"] + measurement_cols
    ].head(10)
)

Merged participant records: 15560
Unique participant IDs: 15560

Values extremely close to zero:
LBXSCR: 0 of 9475 non-NaN values
LBXSBU: 0 of 9473 non-NaN values
URXUMA: 0 of 12510 non-NaN values
URXUCR: 0 of 12509 non-NaN values
URDACT: 0 of 12509 non-NaN values

Measurement summaries before any cleaning:


,count,mean,std,min,1%,5%,50%,95%,99%,max
LBXSCR,9475.0,0.877653,0.490870,0.25,0.4500,0.53,0.82,1.300,1.9200,14.97
LBXSBU,9473.0,14.484746,5.887243,2.00,6.0000,8.00,14.00,24.000,35.0000,79.00
URXUMA,12510.0,46.510537,316.168829,0.21,0.8000,1.70,9.30,110.100,828.3700,16070.00
URXUCR,12509.0,127.240473,85.751110,3.54,13.0000,24.00,111.00,286.000,398.0000,739.00
URDACT,12509.0,44.060051,302.925541,0.27,2.2508,3.18,8.21,97.392,816.6472,11676.92



Participants aged 18 or older: 9693

Adult age range:
18.0 to 80.0

Adult missingness (%):


,0
LBXSCR,14.81
LBXSBU,14.84
URXUMA,9.94
URXUCR,9.95
URDACT,9.95



First 10 rows of relevant measurements:


,SEQN,RIDAGEYR,RIAGENDR,LBXSCR,LBXSBU,URXUMA,URXUCR,URDACT
3,109266.0,29.0,2.0,0.63,8.0,5.5,36.0,15.28
4,109267.0,21.0,2.0,NaN,NaN,NaN,NaN,NaN
5,109268.0,18.0,2.0,NaN,NaN,NaN,NaN,NaN
8,109271.0,49.0,1.0,0.78,8.0,2.4,32.0,7.50
10,109273.0,36.0,1.0,0.92,11.0,4.9,121.0,4.05
11,109274.0,68.0,1.0,0.74,11.0,12.8,120.0,10.67
19,109282.0,76.0,1.0,0.88,18.0,16.0,192.0,8.33
20,109283.0,52.0,1.0,NaN,NaN,NaN,NaN,NaN
21,109284.0,44.0,2.0,0.78,12.0,8.2,189.0,4.34
23,109286.0,33.0,2.0,0.55,9.0,6.8,74.0,9.19


In [7]:
# Load the urine measurements and their laboratory comment codes.
urine_full = pd.read_sas(
    RAW_DATA / "P_ALB_CR.xpt",
    format="xport"
)

print("Urine dataset shape:", urine_full.shape)

comment_cols = ["URDUMALC", "URDUCRLC"]

print("\nDistinct laboratory comment-code values:")
for col in comment_cols:
    print(f"\n{col}:")
    print(urine_full[col].value_counts(dropna=False).head(15))

# Recalculate the albumin-to-creatinine ratio:
# urine albumin (mg/L) / urine creatinine (g/L) = mg/g
ratio_check = urine_full[
    ["URXUMA", "URXUCR", "URDACT"]
].copy()

valid = (
    ratio_check["URXUMA"].notna()
    & ratio_check["URXUCR"].notna()
    & ratio_check["URDACT"].notna()
    & (ratio_check["URXUCR"] > 0)
)

ratio_check = ratio_check.loc[valid].copy()

ratio_check["calculated_ratio"] = (
    ratio_check["URXUMA"] / ratio_check["URXUCR"] * 100
)

ratio_check["absolute_difference"] = (
    ratio_check["calculated_ratio"] - ratio_check["URDACT"]
).abs()

print("\nRecords with all three urine measurements:", len(ratio_check))
print(
    "Maximum absolute difference between calculated and reported ratio:",
    ratio_check["absolute_difference"].max()
)

print("\nRatio comparison summary:")
display(
    ratio_check[
        ["calculated_ratio", "URDACT", "absolute_difference"]
    ].describe().round(4)
)

# Show examples where the ratio differs most.
print("\nFive largest ratio differences:")
display(
    ratio_check.sort_values(
        "absolute_difference", ascending=False
    ).head(5)
)

Urine dataset shape: (13027, 8)

Distinct laboratory comment-code values:

URDUMALC:
URDUMALC
5.397605e-79    12508
NaN               517
1.000000e+00        2
Name: count, dtype: int64

URDUCRLC:
URDUCRLC
5.397605e-79    12507
NaN               518
1.000000e+00        2
Name: count, dtype: int64

Records with all three urine measurements: 12509
Maximum absolute difference between calculated and reported ratio: 0.005000000000109139

Ratio comparison summary:


,calculated_ratio,URDACT,absolute_difference
count,12509.0000,12509.0000,12509.0000
mean,44.0600,44.0601,0.0024
std,302.9255,302.9255,0.0015
min,0.2740,0.2700,0.0000
25%,5.2020,5.2000,0.0011
50%,8.2105,8.2100,0.0024
75%,15.8333,15.8300,0.0036
max,11676.9231,11676.9200,0.0050



Five largest ratio differences:


,URXUMA,URXUCR,URDACT,calculated_ratio,absolute_difference
12501,873.0,32.0,2728.13,2728.125,0.005
8853,72.6,160.0,45.38,45.375,0.005
9731,25.5,80.0,31.88,31.875,0.005
1134,10.2,32.0,31.88,31.875,0.005
11773,9.1,80.0,11.38,11.375,0.005


In [8]:
import numpy as np
import pandas as pd

# Start with the merged audit dataset from Cell 6.
ckd_audit = audit.loc[audit["RIDAGEYR"] >= 18].copy()

# Calculate race-free 2021 CKD-EPI creatinine eGFR.
# NHANES: RIAGENDR 1 = male, 2 = female.
scr = ckd_audit["LBXSCR"]
age = ckd_audit["RIDAGEYR"]
female = ckd_audit["RIAGENDR"].eq(2)
male = ckd_audit["RIAGENDR"].eq(1)

ckd_audit["eGFR_2021"] = np.nan

valid = (
    scr.notna()
    & age.notna()
    & (scr > 0)
    & (age >= 18)
    & (female | male)
)

# Female equation
mask_f = valid & female
ckd_audit.loc[mask_f, "eGFR_2021"] = (
    142
    * np.minimum(scr[mask_f] / 0.7, 1) ** (-0.241)
    * np.maximum(scr[mask_f] / 0.7, 1) ** (-1.200)
    * 0.9938 ** age[mask_f]
    * 1.012
)

# Male equation
mask_m = valid & male
ckd_audit.loc[mask_m, "eGFR_2021"] = (
    142
    * np.minimum(scr[mask_m] / 0.9, 1) ** (-0.302)
    * np.maximum(scr[mask_m] / 0.9, 1) ** (-1.200)
    * 0.9938 ** age[mask_m]
)

# Flag each marker only when its measurement is available.
ckd_audit["low_eGFR_marker"] = (
    ckd_audit["eGFR_2021"] < 60
).where(ckd_audit["eGFR_2021"].notna())

ckd_audit["high_UACR_marker"] = (
    ckd_audit["URDACT"] >= 30
).where(ckd_audit["URDACT"].notna())

# Positive if either observed marker is abnormal.
# Negative only if BOTH markers are observed and neither is abnormal.
any_positive = (
    ckd_audit["low_eGFR_marker"].eq(True)
    | ckd_audit["high_UACR_marker"].eq(True)
)

both_observed = (
    ckd_audit["eGFR_2021"].notna()
    & ckd_audit["URDACT"].notna()
)

ckd_audit["ckd_research_indicator"] = np.nan
ckd_audit.loc[any_positive, "ckd_research_indicator"] = 1
ckd_audit.loc[
    both_observed & ~any_positive, "ckd_research_indicator"
] = 0

print("Adult cohort:", len(ckd_audit))
print("\neGFR available:", ckd_audit["eGFR_2021"].notna().sum())
print("UACR available:", ckd_audit["URDACT"].notna().sum())
print("Both markers available:", both_observed.sum())

print("\neGFR summary:")
display(ckd_audit["eGFR_2021"].describe().round(2))

print("\nMarker counts:")
print("eGFR < 60:", ckd_audit["low_eGFR_marker"].eq(True).sum())
print("UACR >= 30 mg/g:", ckd_audit["high_UACR_marker"].eq(True).sum())

print("\nResearch-indicator counts (1 = either marker positive):")
display(
    ckd_audit["ckd_research_indicator"]
    .value_counts(dropna=False)
    .rename(index={1.0: "Positive", 0.0: "Negative"})
    .to_frame("participants")
)

print("\nResearch-indicator percentages among labelled participants:")
display(
    (
        ckd_audit["ckd_research_indicator"]
        .value_counts(normalize=True)
        .mul(100)
        .round(2)
    ).rename(index={1.0: "Positive", 0.0: "Negative"})
    .to_frame("percent")
)

Adult cohort: 9693

eGFR available: 8257
UACR available: 8729
Both markers available: 8114

eGFR summary:


,eGFR_2021
count,8257.00
mean,94.84
std,23.22
min,2.96
25%,80.22
50%,97.63
75%,112.04
max,163.59



Marker counts:
eGFR < 60: 669
UACR >= 30 mg/g: 1201

Research-indicator counts (1 = either marker positive):


,participants
ckd_research_indicator,
Negative,6644
Positive,1623
NaN,1426



Research-indicator percentages among labelled participants:


,percent
ckd_research_indicator,
Negative,80.37
Positive,19.63


In [9]:
# Cell 9: Target feasibility and consistency audit

print("1. Marker availability")
availability = pd.crosstab(
    ckd_audit["eGFR_2021"].notna(),
    ckd_audit["URDACT"].notna(),
    margins=True
)
availability.index = availability.index.map(str)
availability.columns = availability.columns.map(str)
display(availability)

print("\n2. Marker overlap among participants with both measurements")
both = ckd_audit.loc[both_observed].copy()

overlap = pd.crosstab(
    both["low_eGFR_marker"].map({True: "Low eGFR", False: "eGFR >= 60"}),
    both["high_UACR_marker"].map({True: "UACR >= 30", False: "UACR < 30"}),
    margins=True
)
display(overlap)

print("\n3. Extreme eGFR estimates: lowest 10")
display(
    ckd_audit.loc[
        ckd_audit["eGFR_2021"].notna(),
        ["SEQN", "RIDAGEYR", "RIAGENDR", "LBXSCR", "eGFR_2021", "URDACT"]
    ]
    .nsmallest(10, "eGFR_2021")
)

print("\n4. Creatinine distribution")
display(
    ckd_audit["LBXSCR"]
    .describe(percentiles=[0.01, 0.05, 0.50, 0.95, 0.99])
    .to_frame("serum_creatinine_mg_dl")
    .round(3)
)

print("\n5. Label counts by sex")
display(
    pd.crosstab(
        ckd_audit["RIAGENDR"].map({1: "Male", 2: "Female"}),
        ckd_audit["ckd_research_indicator"].map({
            1.0: "Positive",
            0.0: "Negative"
        }).fillna("Unlabelled"),
        margins=True
    )
)

1. Marker availability


URDACT,False,True,All
eGFR_2021,,,
False,821,615,1436
True,143,8114,8257
All,964,8729,9693



2. Marker overlap among participants with both measurements


high_UACR_marker,UACR < 30,UACR >= 30,All
low_eGFR_marker,,,
Low eGFR,371,247,618
eGFR >= 60,6644,852,7496
All,7015,1099,8114



3. Extreme eGFR estimates: lowest 10


,SEQN,RIDAGEYR,RIAGENDR,LBXSCR,eGFR_2021,URDACT
11393,120656.0,80.0,1.0,14.97,2.958234,1172.26
5749,115012.0,72.0,2.0,10.96,3.383306,NaN
742,110005.0,45.0,2.0,10.60,4.165567,3903.57
9136,118399.0,66.0,1.0,10.78,4.785974,1555.26
15210,124473.0,44.0,2.0,9.48,4.792611,NaN
13810,123073.0,52.0,1.0,11.46,4.851828,NaN
2843,112106.0,30.0,1.0,12.74,4.899433,2361.54
979,110242.0,80.0,1.0,9.11,5.368793,401.27
4401,113664.0,80.0,1.0,8.75,5.634937,NaN
15454,124717.0,51.0,2.0,7.97,5.650477,NaN



4. Creatinine distribution


,serum_creatinine_mg_dl
count,8257.000
mean,0.904
std,0.517
min,0.250
1%,0.460
5%,0.560
50%,0.840
95%,1.330
99%,1.994
max,14.970



5. Label counts by sex


ckd_research_indicator,Negative,Positive,Unlabelled,All
RIAGENDR,,,,
Female,3420,842,713,4975
Male,3224,781,713,4718
All,6644,1623,1426,9693


In [10]:
# Cell 10: Inventory candidate predictor tables and survey-design variables

from pathlib import Path

xpt_paths = sorted(RAW_DATA.glob("*.xpt"))

inventory = []
selected_tables = [
    "P_DEMO.xpt",
    "P_BMX.xpt",
    "P_BPXO.xpt",
    "P_DIQ.xpt",
    "P_HUQ.xpt",
    "P_HIQ.xpt",
    "P_PAQ.xpt",
    "P_SLQ.xpt",
    "P_SMQ.xpt",
    "P_ALQ.xpt",
    "P_DBQ.xpt",
    "P_BIOPRO.xpt",
    "P_GHB.xpt",
    "P_GLU.xpt",
    "P_HDL.xpt",
    "P_TCHOL.xpt",
    "P_HSCRP.xpt",
]

selected_columns = {}

for path in xpt_paths:
    table = pd.read_sas(path, format="xport")
    columns = list(table.columns)

    inventory.append({
        "file": path.name,
        "rows": len(table),
        "columns": len(columns),
        "has_SEQN": "SEQN" in columns,
        "survey_design_columns": [
            c for c in columns
            if any(term in c.upper() for term in
                   ["WTINT", "WTMEC", "WTSAF", "WTDRD", "SDMVSTRA", "SDMVPSU"])
        ],
    })

    if path.name in selected_tables:
        selected_columns[path.name] = columns

    del table

inventory_df = pd.DataFrame(inventory)

print("1. NHANES table inventory")
display(inventory_df)

print("\n2. Survey-weight and design variables found")
design_rows = inventory_df.loc[
    inventory_df["survey_design_columns"].map(len) > 0,
    ["file", "survey_design_columns"]
]
display(design_rows)

print("\n3. Candidate predictor variables by selected table")
for filename, columns in selected_columns.items():
    print(f"\n{filename} ({len(columns)} columns)")
    print(", ".join(columns))

1. NHANES table inventory


,file,rows,columns,has_SEQN,survey_design_columns
0,P_ALB_CR.xpt,13027,8,True,[]
1,P_ALQ.xpt,8965,10,True,[]
2,P_BIOPRO.xpt,10409,41,True,[]
3,P_BMX.xpt,14300,22,True,[]
4,P_BPXO.xpt,11656,12,True,[]
5,P_CBC.xpt,13772,22,True,[]
6,P_DBQ.xpt,15560,46,True,[]
7,P_DEMO.xpt,15560,29,True,"[WTINTPRP, WTMECPRP, SDMVPSU, SDMVSTRA]"
8,P_DIQ.xpt,14986,28,True,[]
9,P_GHB.xpt,10409,2,True,[]



2. Survey-weight and design variables found


,file,survey_design_columns
7,P_DEMO.xpt,"[WTINTPRP, WTMECPRP, SDMVPSU, SDMVSTRA]"
10,P_GLU.xpt,[WTSAFPRP]
20,P_TRIGLY.xpt,[WTSAFPRP]



3. Candidate predictor variables by selected table

P_ALQ.xpt (10 columns)
SEQN, ALQ111, ALQ121, ALQ130, ALQ142, ALQ270, ALQ280, ALQ290, ALQ151, ALQ170

P_BIOPRO.xpt (41 columns)
SEQN, LBXSATSI, LBDSATLC, LBXSAL, LBDSALSI, LBXSAPSI, LBXSASSI, LBXSC3SI, LBXSBU, LBDSBUSI, LBXSCLSI, LBXSCK, LBXSCR, LBDSCRSI, LBXSGB, LBDSGBSI, LBXSGL, LBDSGLSI, LBXSGTSI, LBDSGTLC, LBXSIR, LBDSIRSI, LBXSLDSI, LBXSOSSI, LBXSPH, LBDSPHSI, LBXSKSI, LBXSNASI, LBXSTB, LBDSTBSI, LBDSTBLC, LBXSCA, LBDSCASI, LBXSCH, LBDSCHSI, LBXSTP, LBDSTPSI, LBXSTR, LBDSTRSI, LBXSUA, LBDSUASI

P_BMX.xpt (22 columns)
SEQN, BMDSTATS, BMXWT, BMIWT, BMXRECUM, BMIRECUM, BMXHEAD, BMIHEAD, BMXHT, BMIHT, BMXBMI, BMDBMIC, BMXLEG, BMILEG, BMXARML, BMIARML, BMXARMC, BMIARMC, BMXWAIST, BMIWAIST, BMXHIP, BMIHIP

P_BPXO.xpt (12 columns)
SEQN, BPAOARM, BPAOCSZ, BPXOSY1, BPXODI1, BPXOSY2, BPXODI2, BPXOSY3, BPXODI3, BPXOPLS1, BPXOPLS2, BPXOPLS3

P_DBQ.xpt (46 columns)
SEQN, DBQ010, DBD030, DBD041, DBD050, DBD055, DBD061, DBQ073A, DBQ073B, DBQ073

In [11]:
# Cell 11: Join candidate predictors and audit their availability

candidate_specs = {
    "P_BMX.xpt": [
        "BMXBMI", "BMXWAIST"
    ],
    "P_BPXO.xpt": [
        "BPXOSY1", "BPXOSY2", "BPXOSY3",
        "BPXODI1", "BPXODI2", "BPXODI3"
    ],
    "P_DIQ.xpt": [
        "DIQ010"
    ],
    "P_HIQ.xpt": [
        "HIQ011"
    ],
    "P_HUQ.xpt": [
        "HUQ010"
    ],
    "P_PAQ.xpt": [
        "PAQ605", "PAQ620", "PAQ650", "PAQ665"
    ],
    "P_SMQ.xpt": [
        "SMQ020", "SMQ040"
    ],
    "P_GHB.xpt": [
        "LBXGH"
    ],
}

candidate_data = ckd_audit.copy()

for filename, wanted_columns in candidate_specs.items():
    file_path = RAW_DATA / filename
    table = pd.read_sas(file_path, format="xport")

    available_columns = [
        column for column in wanted_columns
        if column in table.columns
    ]

    table = table[["SEQN"] + available_columns].copy()

    if table["SEQN"].duplicated().any():
        raise ValueError(f"Duplicate SEQN values found in {filename}")

    candidate_data = candidate_data.merge(
        table,
        on="SEQN",
        how="left",
        validate="one_to_one"
    )

print("Adult cohort rows:", len(candidate_data))
print("Unique participant IDs:", candidate_data["SEQN"].nunique())

# Summarize completeness and number of distinct observed values.
candidate_columns = [
    column
    for columns in candidate_specs.values()
    for column in columns
]

feature_audit = pd.DataFrame({
    "variable": candidate_columns,
    "non_missing": [
        candidate_data[column].notna().sum()
        for column in candidate_columns
    ],
    "missing": [
        candidate_data[column].isna().sum()
        for column in candidate_columns
    ],
    "missing_percent": [
        round(candidate_data[column].isna().mean() * 100, 2)
        for column in candidate_columns
    ],
    "unique_non_missing_values": [
        candidate_data[column].nunique(dropna=True)
        for column in candidate_columns
    ],
})

print("\n1. Candidate feature completeness")
display(feature_audit)

print("\n2. Observed value frequencies (top 12 per variable)")
for column in candidate_columns:
    print(f"\n{column}")
    display(
        candidate_data[column]
        .value_counts(dropna=False)
        .head(12)
        .rename_axis("value")
        .to_frame("count")
    )

print("\n3. Survey-design fields retained from the demographic table")
design_columns = [
    column for column in
    ["WTINTPRP", "WTMECPRP", "SDMVSTRA", "SDMVPSU"]
    if column in candidate_data.columns
]
print(design_columns)

Adult cohort rows: 9693
Unique participant IDs: 9693

1. Candidate feature completeness


,variable,non_missing,missing,missing_percent,unique_non_missing_values
0,BMXBMI,8790,903,9.32,452
1,BMXWAIST,8449,1244,12.83,877
2,BPXOSY1,8024,1669,17.22,138
3,BPXOSY2,8013,1680,17.33,138
4,BPXOSY3,7993,1700,17.54,136
5,BPXODI1,8024,1669,17.22,95
6,BPXODI2,8013,1680,17.33,92
7,BPXODI3,7993,1700,17.54,90
8,DIQ010,9693,0,0.00,4
9,HIQ011,9693,0,0.00,4



2. Observed value frequencies (top 12 per variable)

BMXBMI


,count
value,
NaN,903
25.5,70
29.1,70
25.2,70
27.9,68
26.1,65
29.0,63
24.3,63
25.8,62



BMXWAIST


...


SMQ040


,count
value,
NaN,5804
3.0,2205
1.0,1306
2.0,378



LBXGH


,count
value,
NaN,1229
5.4,763
5.5,737
5.3,695
5.6,667
5.7,620
5.2,563
5.8,508
5.1,456



3. Survey-design fields retained from the demographic table
[]


In [12]:
# Cell 12: Inspect candidate-variable coding and survey-design fields

candidate_vars = [
    "RIAGENDR", "RIDAGEYR",
    "DIQ010", "HIQ011", "HUQ010",
    "PAQ605", "PAQ620", "PAQ650", "PAQ665",
    "SMQ020", "SMQ040",
    "BMXBMI", "BMXWAIST",
    "BPXOSY1", "BPXOSY2", "BPXOSY3",
    "BPXODI1", "BPXODI2", "BPXODI3",
    "LBXGH"
]

print("1. Variables present in adult cohort")
print([v for v in candidate_vars if v in ckd_audit.columns])

print("\n2. Special or uncommon response codes")
for col in [
    "DIQ010", "HIQ011", "HUQ010",
    "PAQ605", "PAQ620", "PAQ650", "PAQ665",
    "SMQ020", "SMQ040"
]:
    if col in ckd_audit.columns:
        print(f"\n{col}")
        print(ckd_audit[col].value_counts(dropna=False).sort_index())

print("\n3. Demographic columns relevant to survey design")
survey_candidates = [
    col for col in demo.columns
    if any(term in col.upper() for term in ["WT", "SDMVSTRA", "SDMVPSU"])
]
print(survey_candidates)

print("\n4. Check whether candidate variables are in the merged cohort")
print({
    col: col in ckd_audit.columns
    for col in ["WTMEC2YR", "WTINT2YR", "SDMVSTRA", "SDMVPSU"]
})

1. Variables present in adult cohort
['RIAGENDR', 'RIDAGEYR']

2. Special or uncommon response codes

3. Demographic columns relevant to survey design
[]

4. Check whether candidate variables are in the merged cohort
{'WTMEC2YR': False, 'WTINT2YR': False, 'SDMVSTRA': False, 'SDMVPSU': False}


In [13]:
# Cell 13: Inspect original NHANES columns before merging predictors

import os
import pandas as pd

raw_dir = "/content/Trustworthy-Agentic-AI-for-Healthcare/data/raw"

files_to_check = {
    "P_DEMO.xpt": [
        "SEQN", "RIDAGEYR", "RIAGENDR",
        "WTMEC2YR", "WTINT2YR", "SDMVSTRA", "SDMVPSU"
    ],
    "P_BMX.xpt": ["SEQN", "BMXBMI", "BMXWAIST"],
    "P_BPXO.xpt": [
        "SEQN", "BPXOSY1", "BPXOSY2", "BPXOSY3",
        "BPXODI1", "BPXODI2", "BPXODI3"
    ],
    "P_DIQ.xpt": ["SEQN", "DIQ010"],
    "P_HIQ.xpt": ["SEQN", "HIQ011"],
    "P_HUQ.xpt": ["SEQN", "HUQ010"],
    "P_PAQ.xpt": [
        "SEQN", "PAQ605", "PAQ620", "PAQ650", "PAQ665"
    ],
    "P_SMQ.xpt": ["SEQN", "SMQ020", "SMQ040"],
    "P_GHB.xpt": ["SEQN", "LBXGH"]
}

for filename, expected_vars in files_to_check.items():
    filepath = os.path.join(raw_dir, filename)

    print(f"\n--- {filename} ---")

    if not os.path.exists(filepath):
        print("FILE NOT FOUND")
        continue

    df = pd.read_sas(filepath, format="xport")

    print("Shape:", df.shape)
    print("SEQN present:", "SEQN" in df.columns)
    print("Expected variables found:",
          [v for v in expected_vars if v in df.columns])
    print("Expected variables missing:",
          [v for v in expected_vars if v not in df.columns])

    if "SEQN" in df.columns:
        print("Unique participant IDs:", df["SEQN"].nunique())

print("\n--- Current audit columns ---")
print("Candidate variables present in ckd_audit:")
print([
    v for v in [
        "WTMEC2YR", "WTINT2YR", "SDMVSTRA", "SDMVPSU",
        "BMXBMI", "BMXWAIST", "DIQ010", "LBXGH"
    ]
    if v in ckd_audit.columns
])


--- P_DEMO.xpt ---
Shape: (15560, 29)
SEQN present: True
Expected variables found: ['SEQN', 'RIDAGEYR', 'RIAGENDR', 'SDMVSTRA', 'SDMVPSU']
Expected variables missing: ['WTMEC2YR', 'WTINT2YR']
Unique participant IDs: 15560

--- P_BMX.xpt ---
Shape: (14300, 22)
SEQN present: True
Expected variables found: ['SEQN', 'BMXBMI', 'BMXWAIST']
Expected variables missing: []
Unique participant IDs: 14300

--- P_BPXO.xpt ---
Shape: (11656, 12)
SEQN present: True
Expected variables found: ['SEQN', 'BPXOSY1', 'BPXOSY2', 'BPXOSY3', 'BPXODI1', 'BPXODI2', 'BPXODI3']
Expected variables missing: []
Unique participant IDs: 11656

--- P_DIQ.xpt ---
Shape: (14986, 28)
SEQN present: True
Expected variables found: ['SEQN', 'DIQ010']
Expected variables missing: []
Unique participant IDs: 14986

--- P_HIQ.xpt ---
Shape: (15560, 14)
SEQN present: True
Expected variables found: ['SEQN', 'HIQ011']
Expected variables missing: []
Unique participant IDs: 15560

--- P_HUQ.xpt ---
Shape: (15560, 7)
SEQN present: True


In [14]:
# Cell 14: Merge candidate predictors into a separate adult cohort

import os
import pandas as pd

raw_dir = "/content/Trustworthy-Agentic-AI-for-Healthcare/data/raw"

# Start with the existing adult audit cohort; do not modify ckd_audit
ckd_features = ckd_audit.copy()

# Inspect the actual survey-weight column names in the demographic file
demo_raw = pd.read_sas(
    os.path.join(raw_dir, "P_DEMO.xpt"),
    format="xport"
)

weight_cols = [
    col for col in demo_raw.columns
    if "WT" in col.upper()
]

survey_cols = [
    col for col in ["SDMVSTRA", "SDMVPSU"] + weight_cols
    if col in demo_raw.columns
]

print("Survey-design and weight columns found:", survey_cols)

# Add survey-design fields and available weight variables
demo_add = demo_raw[["SEQN"] + survey_cols].copy()

ckd_features = ckd_features.merge(
    demo_add,
    on="SEQN",
    how="left",
    validate="one_to_one"
)

# Candidate predictor files and the variables to retain
files_and_vars = {
    "P_BMX.xpt": ["BMXBMI", "BMXWAIST"],
    "P_BPXO.xpt": [
        "BPXOSY1", "BPXOSY2", "BPXOSY3",
        "BPXODI1", "BPXODI2", "BPXODI3"
    ],
    "P_DIQ.xpt": ["DIQ010"],
    "P_HIQ.xpt": ["HIQ011"],
    "P_HUQ.xpt": ["HUQ010"],
    "P_PAQ.xpt": ["PAQ605", "PAQ620", "PAQ650", "PAQ665"],
    "P_SMQ.xpt": ["SMQ020", "SMQ040"],
    "P_GHB.xpt": ["LBXGH"]
}

for filename, variables in files_and_vars.items():
    filepath = os.path.join(raw_dir, filename)
    source = pd.read_sas(filepath, format="xport")

    available = [v for v in variables if v in source.columns]
    merge_data = source[["SEQN"] + available].copy()

    # Prevent accidental duplicate column names
    merge_data = merge_data[
        ["SEQN"] + [
            col for col in available
            if col not in ckd_features.columns
        ]
    ]

    ckd_features = ckd_features.merge(
        merge_data,
        on="SEQN",
        how="left",
        validate="one_to_one"
    )

print("\nOriginal adult cohort rows:", len(ckd_audit))
print("Merged cohort rows:", len(ckd_features))
print("Unique participant IDs:", ckd_features["SEQN"].nunique())

print("\nCandidate variable availability:")
for col in [
    "BMXBMI", "BMXWAIST",
    "BPXOSY1", "BPXODI1",
    "DIQ010", "HIQ011", "HUQ010",
    "PAQ605", "PAQ620", "PAQ650", "PAQ665",
    "SMQ020", "SMQ040", "LBXGH"
]:
    if col in ckd_features.columns:
        missing = ckd_features[col].isna().sum()
        print(
            f"{col}: non-missing={ckd_features[col].notna().sum()}, "
            f"missing={missing} ({missing / len(ckd_features) * 100:.2f}%)"
        )

print("\nSurvey-design fields in merged cohort:")
print([
    col for col in ["SDMVSTRA", "SDMVPSU"] + weight_cols
    if col in ckd_features.columns
])

print("\nMerged dataframe shape:", ckd_features.shape)

Survey-design and weight columns found: ['SDMVSTRA', 'SDMVPSU', 'WTINTPRP', 'WTMECPRP']

Original adult cohort rows: 9693
Merged cohort rows: 9693
Unique participant IDs: 9693

Candidate variable availability:
BMXBMI: non-missing=8790, missing=903 (9.32%)
BMXWAIST: non-missing=8449, missing=1244 (12.83%)
BPXOSY1: non-missing=8024, missing=1669 (17.22%)
BPXODI1: non-missing=8024, missing=1669 (17.22%)
DIQ010: non-missing=9693, missing=0 (0.00%)
HIQ011: non-missing=9693, missing=0 (0.00%)
HUQ010: non-missing=9693, missing=0 (0.00%)
PAQ605: non-missing=9693, missing=0 (0.00%)
PAQ620: non-missing=9693, missing=0 (0.00%)
PAQ650: non-missing=9693, missing=0 (0.00%)
PAQ665: non-missing=9693, missing=0 (0.00%)
SMQ020: non-missing=9693, missing=0 (0.00%)
SMQ040: non-missing=3889, missing=5804 (59.88%)
LBXGH: non-missing=8464, missing=1229 (12.68%)

Survey-design fields in merged cohort:
['SDMVSTRA', 'SDMVPSU', 'WTINTPRP', 'WTMECPRP']

Merged dataframe shape: (9693, 34)


In [15]:
# Cell 15: Audit questionnaire codes and survey weights

questionnaire_vars = [
    "DIQ010", "HIQ011", "HUQ010",
    "PAQ605", "PAQ620", "PAQ650", "PAQ665",
    "SMQ020", "SMQ040"
]

print("1. Questionnaire response-code distributions")
for col in questionnaire_vars:
    if col in ckd_features.columns:
        print(f"\n{col}")
        print(
            ckd_features[col]
            .value_counts(dropna=False)
            .sort_index()
            .to_string()
        )

print("\n2. Survey-design field completeness")
for col in ["SDMVSTRA", "SDMVPSU", "WTINTPRP", "WTMECPRP"]:
    if col in ckd_features.columns:
        series = ckd_features[col]
        print(f"\n{col}")
        print("Non-missing:", int(series.notna().sum()))
        print("Missing:", int(series.isna().sum()))
        print("Unique values:", int(series.nunique()))

        if col in ["WTINTPRP", "WTMECPRP"]:
            print("Minimum:", series.min())
            print("Median:", series.median())
            print("Maximum:", series.max())
            print("Non-positive values:", int((series <= 0).sum()))

print("\n3. CKD research-indicator counts")
print(
    ckd_features["CKD_research_indicator"]
    .value_counts(dropna=False)
    .sort_index()
    if "CKD_research_indicator" in ckd_features.columns
    else "Target column name not found in ckd_features; target remains in ckd_audit."
)

1. Questionnaire response-code distributions

DIQ010
DIQ010
1.0    1423
2.0    7998
3.0     267
9.0       5

HIQ011
HIQ011
1.0    8112
2.0    1554
7.0       7
9.0      20

HUQ010
HUQ010
1.0    1126
2.0    2639
3.0    3556
4.0    1957
5.0     405
7.0       6
9.0       4

PAQ605
PAQ605
1.0    2418
2.0    7268
7.0       1
9.0       6

PAQ620
PAQ620
1.0    4187
2.0    5499
9.0       7

PAQ650
PAQ650
1.0    2422
2.0    7271

PAQ665
PAQ665
1.0    3904
2.0    5787
9.0       2

SMQ020
SMQ020
1.0    3889
2.0    5799
7.0       2
9.0       3

SMQ040
SMQ040
1.0    1306
2.0     378
3.0    2205
NaN    5804

2. Survey-design field completeness

SDMVSTRA
Non-missing: 9693
Missing: 0
Unique values: 24

SDMVPSU
Non-missing: 9693
Missing: 0
Unique values: 3

WTINTPRP
Non-missing: 9693
Missing: 0
Unique values: 8958
Minimum: 1876.867755
Median: 15478.338353
Maximum: 338363.600192
Non-positive values: 0

WTMECPRP
Non-missing: 9693
Missing: 0
Unique values: 8367
Minimum: 5.397605346934028e-79
Median: 15301.

In [16]:
# Cell 16: Investigate special values and locate the target column

print("1. Target-related columns in ckd_audit")
print([
    col for col in ckd_audit.columns
    if any(term in col.lower() for term in [
        "ckd", "egfr", "uacr", "albumin", "target", "label"
    ])
])

print("\n2. Extremely small values in survey weights")
for col in ["WTINTPRP", "WTMECPRP"]:
    if col in ckd_features.columns:
        series = ckd_features[col]
        tiny = series[series.abs() < 1e-10]

        print(f"\n{col}")
        print("Count with absolute value < 1e-10:", len(tiny))
        print("Example values:")
        print(tiny.head(10).to_list())

print("\n3. Survey PSU value counts")
if "SDMVPSU" in ckd_features.columns:
    print(
        ckd_features["SDMVPSU"]
        .value_counts(dropna=False)
        .sort_index()
        .to_string()
    )

print("\n4. Check target columns in both dataframes")
for name, df in [
    ("ckd_audit", ckd_audit),
    ("ckd_features", ckd_features)
]:
    print(f"\n{name}:")
    for col in df.columns:
        if any(term in col.lower() for term in [
            "ckd", "egfr", "uacr", "target", "label"
        ]):
            print(f"  {col}: {df[col].value_counts(dropna=False).head(8).to_dict()}")

print("\n5. Confirm whether original weight values also contain tiny values")
demo_raw = pd.read_sas(
    os.path.join(raw_dir, "P_DEMO.xpt"),
    format="xport"
)

for col in ["WTINTPRP", "WTMECPRP"]:
    if col in demo_raw.columns:
        count = int((demo_raw[col].abs() < 1e-10).sum())
        print(f"{col}: {count} original records with absolute value < 1e-10")

1. Target-related columns in ckd_audit
['eGFR_2021', 'low_eGFR_marker', 'high_UACR_marker', 'ckd_research_indicator']

2. Extremely small values in survey weights

WTINTPRP
Count with absolute value < 1e-10: 0
Example values:
[]

WTMECPRP
Count with absolute value < 1e-10: 728
Example values:
[5.397605346934028e-79, 5.397605346934028e-79, 5.397605346934028e-79, 5.397605346934028e-79, 5.397605346934028e-79, 5.397605346934028e-79, 5.397605346934028e-79, 5.397605346934028e-79, 5.397605346934028e-79, 5.397605346934028e-79]

3. Survey PSU value counts
SDMVPSU
1.0    4633
2.0    4837
3.0     223

4. Check target columns in both dataframes

ckd_audit:
  eGFR_2021: {nan: 1436, 73.33712406952934: 10, 68.60992600459812: 9, 85.90053708048798: 9, 84.09174325259: 9, 71.75820881702404: 8, 75.57069009474026: 8, 97.2943389688813: 7}
  low_eGFR_marker: {False: 7588, nan: 1436, True: 669}
  high_UACR_marker: {False: 7528, True: 1201, nan: 964}
  ckd_research_indicator: {0.0: 6644, 1.0: 1623, nan: 1426}


In [17]:
# Cell 17: Inspect NHANES demographic file metadata and anomalous weights

import os
import pandas as pd

demo_path = os.path.join(raw_dir, "P_DEMO.xpt")

print("1. pandas version:", pd.__version__)

print("\n2. Original weight values around zero")
for col in ["WTINTPRP", "WTMECPRP"]:
    if col in demo_raw.columns:
        values = demo_raw[col].sort_values()
        print(f"\n{col} — 15 smallest values:")
        print(values.head(15).to_list())

print("\n3. Original survey design values")
for col in ["SDMVSTRA", "SDMVPSU"]:
    if col in demo_raw.columns:
        print(f"\n{col}:")
        print(
            demo_raw[col]
            .value_counts(dropna=False)
            .sort_index()
            .to_string()
        )

print("\n4. Weight summary, excluding extremely small values")
for col in ["WTINTPRP", "WTMECPRP"]:
    if col in demo_raw.columns:
        series = demo_raw[col]
        valid = series[series.abs() >= 1e-10]

        print(f"\n{col}")
        print("Original rows:", len(series))
        print("Values below 1e-10 in absolute magnitude:",
              int((series.abs() < 1e-10).sum()))
        print("Remaining values:", len(valid))
        print("Minimum remaining value:", valid.min())
        print("Median remaining value:", valid.median())
        print("Maximum remaining value:", valid.max())

print("\n5. Demographic file columns")
print(demo_raw.columns.to_list())

1. pandas version: 2.2.3

2. Original weight values around zero

WTINTPRP — 15 smallest values:
[1017.784203, 1241.066032, 1325.007813, 1415.458133, 1422.23066, 1436.522098, 1439.623468, 1465.79844, 1509.624738, 1513.711657, 1514.886008, 1554.317975, 1554.317975, 1554.736553, 1602.708771]

WTMECPRP — 15 smallest values:
[5.397605346934028e-79, 5.397605346934028e-79, 5.397605346934028e-79, 5.397605346934028e-79, 5.397605346934028e-79, 5.397605346934028e-79, 5.397605346934028e-79, 5.397605346934028e-79, 5.397605346934028e-79, 5.397605346934028e-79, 5.397605346934028e-79, 5.397605346934028e-79, 5.397605346934028e-79, 5.397605346934028e-79, 5.397605346934028e-79]

3. Original survey design values

SDMVSTRA:
SDMVSTRA
149.0     691
150.0     701
151.0     779
152.0     681
153.0     510
154.0     510
155.0     638
156.0    1015
157.0     653
158.0     735
159.0     760
160.0     324
161.0     695
162.0     554
163.0     609
164.0     693
165.0     612
166.0     551
167.0     604
168.0     69

In [18]:
# Cell 18: Inspect SAS/XPORT metadata using pyreadstat, if available

try:
    import pyreadstat

    print("pyreadstat version:", pyreadstat.__version__)

    meta_df, meta = pyreadstat.read_xport(demo_path)

    print("\nData shape:", meta_df.shape)
    print("\nWeight and survey-design variables:")
    for col in ["WTINTPRP", "WTMECPRP", "SDMVSTRA", "SDMVPSU"]:
        if col in meta_df.columns:
            print(f"\n{col}")
            print(meta_df[col].value_counts(dropna=False).sort_index().head(20))

    print("\nColumn labels:")
    labels = getattr(meta, "column_names_to_labels", {})
    for col in ["WTINTPRP", "WTMECPRP", "SDMVSTRA", "SDMVPSU"]:
        print(col, "->", labels.get(col, "No label available"))

    print("\nMissing-value metadata:")
    missing_info = getattr(meta, "missing_ranges", None)
    print(missing_info if missing_info else "No missing ranges exposed")

except ImportError:
    print("pyreadstat is not installed in this Colab environment.")
    print("No package was installed; the original data remains unchanged.")

pyreadstat is not installed in this Colab environment.
No package was installed; the original data remains unchanged.


In [19]:
# Cell 19: Audit target definition and potential predictor leakage

target_col = "ckd_research_indicator"

target_measurements = [
    "LBXSCR",       # Serum creatinine
    "eGFR_2021",    # Calculated from serum creatinine, age and sex
    "URDACT",       # Urine albumin-creatinine ratio
    "URXUMA",       # Urine albumin
    "URXUCR",       # Urine creatinine
    "low_eGFR_marker",
    "high_UACR_marker"
]

print("1. Target availability in adult audit")
print("Target column present:", target_col in ckd_audit.columns)

if target_col in ckd_audit.columns:
    print(ckd_audit[target_col].value_counts(dropna=False).sort_index())

print("\n2. Target-defining measurements present in candidate dataset")
print({
    col: col in ckd_features.columns
    for col in target_measurements
})

print("\n3. Candidate predictors excluding direct target measurements")
candidate_predictors = [
    "RIDAGEYR", "RIAGENDR",
    "BMXBMI", "BMXWAIST",
    "BPXOSY1", "BPXOSY2", "BPXOSY3",
    "BPXODI1", "BPXODI2", "BPXODI3",
    "DIQ010", "HIQ011", "HUQ010",
    "PAQ605", "PAQ620", "PAQ650", "PAQ665",
    "SMQ020", "SMQ040", "LBXGH"
]

available_predictors = [
    col for col in candidate_predictors
    if col in ckd_features.columns
]

print(available_predictors)

print("\n4. Check predictor overlap with target-defining measurements")
print([
    col for col in available_predictors
    if col in target_measurements
])

print("\n5. Cohort and target counts")
print("Adult rows:", len(ckd_features))
print("Unique IDs:", ckd_features["SEQN"].nunique())

if target_col in ckd_audit.columns:
    print("Labeled participants:", int(ckd_audit[target_col].notna().sum()))
    print("Unlabeled participants:", int(ckd_audit[target_col].isna().sum()))

1. Target availability in adult audit
Target column present: True
ckd_research_indicator
0.0    6644
1.0    1623
NaN    1426
Name: count, dtype: int64

2. Target-defining measurements present in candidate dataset
{'LBXSCR': True, 'eGFR_2021': True, 'URDACT': True, 'URXUMA': True, 'URXUCR': True, 'low_eGFR_marker': True, 'high_UACR_marker': True}

3. Candidate predictors excluding direct target measurements
['RIDAGEYR', 'RIAGENDR', 'BMXBMI', 'BMXWAIST', 'BPXOSY1', 'BPXOSY2', 'BPXOSY3', 'BPXODI1', 'BPXODI2', 'BPXODI3', 'DIQ010', 'HIQ011', 'HUQ010', 'PAQ605', 'PAQ620', 'PAQ650', 'PAQ665', 'SMQ020', 'SMQ040', 'LBXGH']

4. Check predictor overlap with target-defining measurements
[]

5. Cohort and target counts
Adult rows: 9693
Unique IDs: 9693
Labeled participants: 8267
Unlabeled participants: 1426


In [20]:
# Cell 20: Prepare a separate labelled dataset for modelling audit

target_col = "ckd_research_indicator"

candidate_predictors = [
    "RIDAGEYR", "RIAGENDR",
    "BMXBMI", "BMXWAIST",
    "BPXOSY1", "BPXOSY2", "BPXOSY3",
    "BPXODI1", "BPXODI2", "BPXODI3",
    "DIQ010", "HIQ011", "HUQ010",
    "PAQ605", "PAQ620", "PAQ650", "PAQ665",
    "SMQ020", "SMQ040", "LBXGH"
]

# Keep only available predictors and the target
available_predictors = [
    col for col in candidate_predictors
    if col in ckd_features.columns
]

model_audit = ckd_features.loc[
    ckd_features[target_col].notna(),
    ["SEQN"] + available_predictors
].copy()

# Attach the target from the original audit by participant ID
target_lookup = ckd_audit[["SEQN", target_col]].copy()

model_audit = model_audit.merge(
    target_lookup,
    on="SEQN",
    how="left",
    validate="one_to_one"
)

print("Model-audit dataset shape:", model_audit.shape)
print("Unique participant IDs:", model_audit["SEQN"].nunique())

print("\nTarget distribution:")
print(model_audit[target_col].value_counts().sort_index())

print("\nMissingness in candidate predictors:")
missingness = (
    model_audit[available_predictors]
    .isna()
    .sum()
    .to_frame("missing_count")
)
missingness["missing_percent"] = (
    missingness["missing_count"] / len(model_audit) * 100
)
print(missingness.sort_values("missing_percent", ascending=False).round(2))

print("\nDirect target-defining measurements included as predictors:")
print([
    col for col in available_predictors
    if col in [
        "LBXSCR", "eGFR_2021", "URDACT",
        "URXUMA", "URXUCR",
        "low_eGFR_marker", "high_UACR_marker"
    ]
])

print("\nOriginal audit unchanged in row count:", len(ckd_audit) == 9693)

Model-audit dataset shape: (8267, 22)
Unique participant IDs: 8267

Target distribution:
ckd_research_indicator
0.0    6644
1.0    1623
Name: count, dtype: int64

Missingness in candidate predictors:
          missing_count  missing_percent
SMQ040             4938            59.73
BPXOSY3             811             9.81
BPXODI3             811             9.81
BPXODI2             793             9.59
BPXOSY2             793             9.59
BPXODI1             783             9.47
BPXOSY1             783             9.47
BMXWAIST            367             4.44
BMXBMI              120             1.45
LBXGH                81             0.98
RIDAGEYR              0             0.00
RIAGENDR              0             0.00
HIQ011                0             0.00
DIQ010                0             0.00
HUQ010                0             0.00
PAQ605                0             0.00
PAQ650                0             0.00
PAQ620                0             0.00
SMQ020               

In [21]:
# Cell 21: Inspect smoking-questionnaire skip patterns

print("1. SMQ020 distribution")
print(model_audit["SMQ020"].value_counts(dropna=False).sort_index())

print("\n2. SMQ040 distribution")
print(model_audit["SMQ040"].value_counts(dropna=False).sort_index())

print("\n3. SMQ020 versus SMQ040 cross-tabulation")
print(
    pd.crosstab(
        model_audit["SMQ020"],
        model_audit["SMQ040"],
        dropna=False,
        margins=True
    )
)

print("\n4. Missing SMQ040 by SMQ020 response")
smoking_missing_summary = (
    model_audit.assign(SMQ040_missing=model_audit["SMQ040"].isna())
    .groupby("SMQ020", dropna=False)["SMQ040_missing"]
    .agg(["count", "sum", "mean"])
)
smoking_missing_summary["missing_percent"] = (
    smoking_missing_summary["mean"] * 100
)
print(smoking_missing_summary.round(3))

print("\n5. Confirm no recoding has occurred")
print("SMQ020 unique values:", sorted(model_audit["SMQ020"].dropna().unique()))
print("SMQ040 unique non-missing values:", sorted(model_audit["SMQ040"].dropna().unique()))

1. SMQ020 distribution
SMQ020
1.0    3329
2.0    4934
7.0       2
9.0       2
Name: count, dtype: int64

2. SMQ040 distribution
SMQ040
1.0    1117
2.0     323
3.0    1889
NaN    4938
Name: count, dtype: int64

3. SMQ020 versus SMQ040 cross-tabulation
SMQ040   1.0  2.0   3.0   NaN   All
SMQ020                             
1.0     1117  323  1889     0  3329
2.0        0    0     0  4934  4934
7.0        0    0     0     2     2
9.0        0    0     0     2     2
All     1117  323  1889     0  8267

4. Missing SMQ040 by SMQ020 response
        count   sum  mean  missing_percent
SMQ020                                    
1.0      3329     0   0.0              0.0
2.0      4934  4934   1.0            100.0
7.0         2     2   1.0            100.0
9.0         2     2   1.0            100.0

5. Confirm no recoding has occurred
SMQ020 unique values: [np.float64(1.0), np.float64(2.0), np.float64(7.0), np.float64(9.0)]
SMQ040 unique non-missing values: [np.float64(1.0), np.float64(2.0), np.f

In [22]:
# Cell 22: Summarise questionnaire codes before documented recoding

questionnaire_vars = [
    "DIQ010", "HIQ011", "HUQ010",
    "PAQ605", "PAQ620", "PAQ650", "PAQ665",
    "SMQ020"
]

for col in questionnaire_vars:
    print(f"\n--- {col} ---")
    print("Counts:")
    print(model_audit[col].value_counts(dropna=False).sort_index().to_dict())

print("\n--- SMQ040 (conditional smoking question; excluded initially) ---")
print(model_audit["SMQ040"].value_counts(dropna=False).sort_index().to_dict())

print("\nDataset dimensions before recoding:", model_audit.shape)
print("Target counts:")
print(model_audit["ckd_research_indicator"].value_counts().sort_index())


--- DIQ010 ---
Counts:
{1.0: 1245, 2.0: 6779, 3.0: 239, 9.0: 4}

--- HIQ011 ---
Counts:
{1.0: 6930, 2.0: 1318, 7.0: 5, 9.0: 14}

--- HUQ010 ---
Counts:
{1.0: 934, 2.0: 2249, 3.0: 3059, 4.0: 1694, 5.0: 322, 7.0: 5, 9.0: 4}

--- PAQ605 ---
Counts:
{1.0: 2102, 2.0: 6159, 7.0: 1, 9.0: 5}

--- PAQ620 ---
Counts:
{1.0: 3634, 2.0: 4626, 9.0: 7}

--- PAQ650 ---
Counts:
{1.0: 2073, 2.0: 6194}

--- PAQ665 ---
Counts:
{1.0: 3398, 2.0: 4867, 9.0: 2}

--- SMQ020 ---
Counts:
{1.0: 3329, 2.0: 4934, 7.0: 2, 9.0: 2}

--- SMQ040 (conditional smoking question; excluded initially) ---
{1.0: 1117, 2.0: 323, 3.0: 1889, nan: 4938}

Dataset dimensions before recoding: (8267, 22)
Target counts:
ckd_research_indicator
0.0    6644
1.0    1623
Name: count, dtype: int64


In [23]:
# Cell 23: Create a separate copy with special questionnaire codes recoded

model_preprocessed = model_audit.copy()

# Based on the NHANES questionnaire coding conventions for these variables:
# 7 = Refused; 9 = Don't know, where those codes apply.
# Do not apply a code unless it is observed for that specific variable.
special_codes = {
    "DIQ010": [7, 9],
    "HIQ011": [7, 9],
    "HUQ010": [7, 9],
    "PAQ605": [7, 9],
    "PAQ620": [7, 9],
    "PAQ650": [7, 9],
    "PAQ665": [7, 9],
    "SMQ020": [7, 9]
}

print("Before recoding:")
for col, codes in special_codes.items():
    if col in model_preprocessed.columns:
        print(col, model_preprocessed[col].value_counts(dropna=False).sort_index().to_dict())

for col, codes in special_codes.items():
    if col in model_preprocessed.columns:
        model_preprocessed[col] = model_preprocessed[col].replace(codes, float("nan"))

print("\nAfter recoding:")
for col in special_codes:
    if col in model_preprocessed.columns:
        print(col, model_preprocessed[col].value_counts(dropna=False).sort_index().to_dict())

# Exclude conditional smoking frequency from the initial predictor set.
initial_predictors = [
    col for col in available_predictors
    if col != "SMQ040"
]

print("\nInitial predictor list:")
print(initial_predictors)

print("\nDataset shape unchanged:", model_preprocessed.shape == model_audit.shape)
print("Original model_audit unchanged:",
      model_audit["SMQ020"].isin([7, 9]).sum() == 4)

Before recoding:
DIQ010 {1.0: 1245, 2.0: 6779, 3.0: 239, 9.0: 4}
HIQ011 {1.0: 6930, 2.0: 1318, 7.0: 5, 9.0: 14}
HUQ010 {1.0: 934, 2.0: 2249, 3.0: 3059, 4.0: 1694, 5.0: 322, 7.0: 5, 9.0: 4}
PAQ605 {1.0: 2102, 2.0: 6159, 7.0: 1, 9.0: 5}
PAQ620 {1.0: 3634, 2.0: 4626, 9.0: 7}
PAQ650 {1.0: 2073, 2.0: 6194}
PAQ665 {1.0: 3398, 2.0: 4867, 9.0: 2}
SMQ020 {1.0: 3329, 2.0: 4934, 7.0: 2, 9.0: 2}

After recoding:
DIQ010 {1.0: 1245, 2.0: 6779, 3.0: 239, nan: 4}
HIQ011 {1.0: 6930, 2.0: 1318, nan: 19}
HUQ010 {1.0: 934, 2.0: 2249, 3.0: 3059, 4.0: 1694, 5.0: 322, nan: 9}
PAQ605 {1.0: 2102, 2.0: 6159, nan: 6}
PAQ620 {1.0: 3634, 2.0: 4626, nan: 7}
PAQ650 {1.0: 2073, 2.0: 6194}
PAQ665 {1.0: 3398, 2.0: 4867, nan: 2}
SMQ020 {1.0: 3329, 2.0: 4934, nan: 4}

Initial predictor list:
['RIDAGEYR', 'RIAGENDR', 'BMXBMI', 'BMXWAIST', 'BPXOSY1', 'BPXOSY2', 'BPXOSY3', 'BPXODI1', 'BPXODI2', 'BPXODI3', 'DIQ010', 'HIQ011', 'HUQ010', 'PAQ605', 'PAQ620', 'PAQ650', 'PAQ665', 'SMQ020', 'LBXGH']

Dataset shape unchanged: True


In [24]:
# Cell 24: Summarise missingness in the preprocessed candidate predictors

print("Preprocessed dataset shape:", model_preprocessed.shape)

print("\nTarget distribution:")
print(
    model_preprocessed["ckd_research_indicator"]
    .value_counts(dropna=False)
    .sort_index()
)

missing_summary = pd.DataFrame({
    "missing_count": model_preprocessed[initial_predictors].isna().sum(),
    "missing_percent": (
        model_preprocessed[initial_predictors].isna().mean() * 100
    )
}).sort_values("missing_percent", ascending=False)

print("\nMissingness by initial predictor:")
print(missing_summary.round(2).to_string())

print("\nRows with at least one missing predictor:")
rows_with_missing = model_preprocessed[initial_predictors].isna().any(axis=1)
print(int(rows_with_missing.sum()), f"({rows_with_missing.mean() * 100:.2f}%)")

print("\nRows with complete predictors:")
print(int((~rows_with_missing).sum()))

print("\nOriginal audit cohort still intact:", len(ckd_audit) == 9693)
print("Original model_audit still intact:", model_audit.shape == (8267, 22))

Preprocessed dataset shape: (8267, 22)

Target distribution:
ckd_research_indicator
0.0    6644
1.0    1623
Name: count, dtype: int64

Missingness by initial predictor:
          missing_count  missing_percent
BPXOSY3             811             9.81
BPXODI3             811             9.81
BPXODI2             793             9.59
BPXOSY2             793             9.59
BPXODI1             783             9.47
BPXOSY1             783             9.47
BMXWAIST            367             4.44
BMXBMI              120             1.45
LBXGH                81             0.98
HIQ011               19             0.23
HUQ010                9             0.11
PAQ620                7             0.08
PAQ605                6             0.07
DIQ010                4             0.05
SMQ020                4             0.05
PAQ665                2             0.02
RIDAGEYR              0             0.00
RIAGENDR              0             0.00
PAQ650                0             0.00

Rows with 

In [25]:
# Cell 25: Inspect the number of available blood-pressure readings

systolic_cols = ["BPXOSY1", "BPXOSY2", "BPXOSY3"]
diastolic_cols = ["BPXODI1", "BPXODI2", "BPXODI3"]

print("1. Systolic readings available per participant")
sys_count = model_preprocessed[systolic_cols].notna().sum(axis=1)
print(sys_count.value_counts().sort_index().to_dict())

print("\n2. Diastolic readings available per participant")
dia_count = model_preprocessed[diastolic_cols].notna().sum(axis=1)
print(dia_count.value_counts().sort_index().to_dict())

print("\n3. Systolic readings: summary")
print(model_preprocessed[systolic_cols].describe().round(2))

print("\n4. Diastolic readings: summary")
print(model_preprocessed[diastolic_cols].describe().round(2))

print("\n5. Participants with no blood-pressure readings")
no_bp = (sys_count == 0) | (dia_count == 0)
print(int(no_bp.sum()))

print("\n6. Participants with all three readings in both groups")
all_three = (sys_count == 3) & (dia_count == 3)
print(int(all_three.sum()))

print("\n7. Confirm source readings remain unchanged")
print("Original candidate data shape:", model_audit.shape)
print("Preprocessed data shape:", model_preprocessed.shape)

1. Systolic readings available per participant
{0: 783, 1: 10, 2: 18, 3: 7456}

2. Diastolic readings available per participant
{0: 783, 1: 10, 2: 18, 3: 7456}

3. Systolic readings: summary
       BPXOSY1  BPXOSY2  BPXOSY3
count  7484.00  7474.00  7456.00
mean    124.36   124.05   124.03
std      19.66    19.48    19.33
min      66.00    54.00    60.00
25%     110.00   110.00   110.00
50%     121.00   121.00   121.00
75%     135.00   135.00   135.00
max     224.00   222.00   220.00

4. Diastolic readings: summary
       BPXODI1  BPXODI2  BPXODI3
count  7484.00  7474.00  7456.00
mean     74.78    74.22    73.94
std      11.83    11.82    11.84
min      36.00    31.00    36.00
25%      67.00    66.00    66.00
50%      74.00    73.00    73.00
75%      82.00    81.00    81.00
max     141.00   146.00   145.00

5. Participants with no blood-pressure readings
783

6. Participants with all three readings in both groups
7456

7. Confirm source readings remain unchanged
Original candidate data 

In [26]:
# Cell 26: Create mean blood-pressure features without changing source data

model_features = model_preprocessed.copy()

systolic_cols = ["BPXOSY1", "BPXOSY2", "BPXOSY3"]
diastolic_cols = ["BPXODI1", "BPXODI2", "BPXODI3"]

# Average the available readings; all-missing rows remain NaN
model_features["mean_systolic_bp"] = (
    model_features[systolic_cols].mean(axis=1, skipna=True)
)

model_features["mean_diastolic_bp"] = (
    model_features[diastolic_cols].mean(axis=1, skipna=True)
)

# Remove the six individual readings from this new modelling dataset only
model_features = model_features.drop(
    columns=systolic_cols + diastolic_cols
)

print("Original model_audit shape:", model_audit.shape)
print("Preprocessed dataset shape:", model_preprocessed.shape)
print("New feature dataset shape:", model_features.shape)

print("\nNew blood-pressure feature summary:")
print(
    model_features[
        ["mean_systolic_bp", "mean_diastolic_bp"]
    ].describe().round(2)
)

print("\nMissing values in new blood-pressure features:")
print(
    model_features[
        ["mean_systolic_bp", "mean_diastolic_bp"]
    ].isna().sum()
)

print("\nTarget distribution preserved:")
print(
    model_features["ckd_research_indicator"]
    .value_counts(dropna=False)
    .sort_index()
)

print("\nOriginal model_audit unchanged:", model_audit.shape == (8267, 22))

Original model_audit shape: (8267, 22)
Preprocessed dataset shape: (8267, 22)
New feature dataset shape: (8267, 18)

New blood-pressure feature summary:
       mean_systolic_bp  mean_diastolic_bp
count           7484.00            7484.00
mean             124.17              74.31
std               19.15              11.54
min               76.33              41.33
25%              110.33              66.33
50%              121.33              73.67
75%              134.67              81.33
max              218.67             143.67

Missing values in new blood-pressure features:
mean_systolic_bp     783
mean_diastolic_bp    783
dtype: int64

Target distribution preserved:
ckd_research_indicator
0.0    6644
1.0    1623
Name: count, dtype: int64

Original model_audit unchanged: True


In [27]:
# Cell 27: Define and validate candidate predictors

target_col = "ckd_research_indicator"
id_col = "SEQN"

candidate_predictors = [
    "RIDAGEYR",
    "RIAGENDR",
    "BMXBMI",
    "BMXWAIST",
    "mean_systolic_bp",
    "mean_diastolic_bp",
    "DIQ010",
    "HIQ011",
    "HUQ010",
    "PAQ605",
    "PAQ620",
    "PAQ650",
    "PAQ665",
    "SMQ020",
    "LBXGH",
]

# Verify all required columns exist
required_cols = [id_col, target_col] + candidate_predictors
missing_cols = [col for col in required_cols if col not in model_features.columns]

print("Missing required columns:", missing_cols)
print("Number of candidate predictors:", len(candidate_predictors))

# Ensure target-defining measurements are not predictors
target_defining_cols = [
    "LBXSCR",
    "eGFR_2021",
    "URDACT",
    "URXUMA",
    "URXUCR",
    "low_eGFR_marker",
    "high_UACR_marker",
]

overlap = sorted(set(candidate_predictors) & set(target_defining_cols))
print("Target-defining leakage overlap:", overlap)

print("\nCandidate predictor data types:")
print(model_features[candidate_predictors].dtypes.to_string())

print("\nFinal candidate dataset shape:")
print(model_features[[id_col] + candidate_predictors + [target_col]].shape)

print("\nUnique participant IDs:", model_features[id_col].nunique())
print("Missing target labels:", int(model_features[target_col].isna().sum()))

assert not missing_cols, f"Missing required columns: {missing_cols}"
assert not overlap, f"Potential target leakage: {overlap}"
assert model_features[id_col].is_unique, "Duplicate participant IDs detected"
assert model_features[target_col].notna().all(), "Unexpected missing target labels"

print("\nAll structural checks passed.")

Missing required columns: []
Number of candidate predictors: 15
Target-defining leakage overlap: []

Candidate predictor data types:
RIDAGEYR             float64
RIAGENDR             float64
BMXBMI               float64
BMXWAIST             float64
mean_systolic_bp     float64
mean_diastolic_bp    float64
DIQ010               float64
HIQ011               float64
HUQ010               float64
PAQ605               float64
PAQ620               float64
PAQ650               float64
PAQ665               float64
SMQ020               float64
LBXGH                float64

Final candidate dataset shape:
(8267, 17)

Unique participant IDs: 8267
Missing target labels: 0

All structural checks passed.


In [28]:
# Cell 28: Document candidate feature meanings and types

feature_documentation = pd.DataFrame([
    ("RIDAGEYR", "Age in years", "Numeric"),
    ("RIAGENDR", "Sex recorded in NHANES demographics", "Categorical"),
    ("BMXBMI", "Body mass index", "Numeric"),
    ("BMXWAIST", "Waist circumference", "Numeric"),
    ("mean_systolic_bp", "Mean of available systolic blood-pressure readings", "Numeric"),
    ("mean_diastolic_bp", "Mean of available diastolic blood-pressure readings", "Numeric"),
    ("DIQ010", "Ever told by a health professional that the participant has diabetes", "Categorical"),
    ("HIQ011", "Health insurance coverage", "Categorical"),
    ("HUQ010", "Self-reported general health status", "Ordinal categorical"),
    ("PAQ605", "Vigorous work-related physical activity", "Categorical"),
    ("PAQ620", "Moderate work-related physical activity", "Categorical"),
    ("PAQ650", "Vigorous recreational physical activity", "Categorical"),
    ("PAQ665", "Moderate recreational physical activity", "Categorical"),
    ("SMQ020", "Ever smoked at least 100 cigarettes", "Categorical"),
    ("LBXGH", "Glycohemoglobin (HbA1c)", "Numeric"),
], columns=["feature", "meaning", "proposed_type"])

print(feature_documentation.to_string(index=False))

print("\nFeature type counts:")
print(feature_documentation["proposed_type"].value_counts())

print("\nDocumentation rows:", len(feature_documentation))
print("All documented features present:",
      set(feature_documentation["feature"]).issubset(model_features.columns))

          feature                                                              meaning       proposed_type
         RIDAGEYR                                                         Age in years             Numeric
         RIAGENDR                                  Sex recorded in NHANES demographics         Categorical
           BMXBMI                                                      Body mass index             Numeric
         BMXWAIST                                                  Waist circumference             Numeric
 mean_systolic_bp                   Mean of available systolic blood-pressure readings             Numeric
mean_diastolic_bp                  Mean of available diastolic blood-pressure readings             Numeric
           DIQ010 Ever told by a health professional that the participant has diabetes         Categorical
           HIQ011                                            Health insurance coverage         Categorical
           HUQ010                    

In [29]:
# Cell 29: Inspect source variable labels

import pandas as pd
from pathlib import Path

raw_dir = Path("/content/Trustworthy-Agentic-AI-for-Healthcare/data/raw")

files_to_check = {
    "P_DEMO.xpt": ["RIAGENDR", "RIDAGEYR"],
    "P_DIQ.xpt": ["DIQ010"],
    "P_HIQ.xpt": ["HIQ011"],
    "P_HUQ.xpt": ["HUQ010"],
    "P_PAQ.xpt": ["PAQ605", "PAQ620", "PAQ650", "PAQ665"],
    "P_SMQ.xpt": ["SMQ020"],
    "P_GHB.xpt": ["LBXGH"],
    "P_BMX.xpt": ["BMXBMI", "BMXWAIST"],
    "P_BPXO.xpt": [
        "BPXOSY1", "BPXOSY2", "BPXOSY3",
        "BPXODI1", "BPXODI2", "BPXODI3"
    ],
}

for filename, variables in files_to_check.items():
    file_path = raw_dir / filename
    print(f"\n--- {filename} ---")

    if not file_path.exists():
        print("File not found:", file_path)
        continue

    # pandas reads the XPT data; variable labels are inspected separately below.
    df = pd.read_sas(file_path, format="xport", encoding="utf-8")

    for variable in variables:
        if variable in df.columns:
            print(f"{variable}: present")
        else:
            print(f"{variable}: NOT FOUND")

print("\nNote: this checks variable names, not the full official response-code definitions.")


--- P_DEMO.xpt ---
RIAGENDR: present
RIDAGEYR: present

--- P_DIQ.xpt ---
DIQ010: present

--- P_HIQ.xpt ---
HIQ011: present

--- P_HUQ.xpt ---
HUQ010: present

--- P_PAQ.xpt ---
PAQ605: present
PAQ620: present
PAQ650: present
PAQ665: present

--- P_SMQ.xpt ---
SMQ020: present

--- P_GHB.xpt ---
LBXGH: present

--- P_BMX.xpt ---
BMXBMI: present
BMXWAIST: present

--- P_BPXO.xpt ---
BPXOSY1: present
BPXOSY2: present
BPXOSY3: present
BPXODI1: present
BPXODI2: present
BPXODI3: present

Note: this checks variable names, not the full official response-code definitions.


In [30]:
# Cell 30: Inspect observed codes for questionnaire predictors

questionnaire_cols = [
    "DIQ010", "HIQ011", "HUQ010",
    "PAQ605", "PAQ620", "PAQ650", "PAQ665", "SMQ020"
]

for col in questionnaire_cols:
    print(f"\n{col}")
    print(model_features[col].value_counts(dropna=False).sort_index().to_string())

print("\nCurrent feature documentation:")
print(feature_documentation.to_string(index=False))


DIQ010
DIQ010
1.0    1245
2.0    6779
3.0     239
NaN       4

HIQ011
HIQ011
1.0    6930
2.0    1318
NaN      19

HUQ010
HUQ010
1.0     934
2.0    2249
3.0    3059
4.0    1694
5.0     322
NaN       9

PAQ605
PAQ605
1.0    2102
2.0    6159
NaN       6

PAQ620
PAQ620
1.0    3634
2.0    4626
NaN       7

PAQ650
PAQ650
1.0    2073
2.0    6194

PAQ665
PAQ665
1.0    3398
2.0    4867
NaN       2

SMQ020
SMQ020
1.0    3329
2.0    4934
NaN       4

Current feature documentation:
          feature                                                              meaning       proposed_type
         RIDAGEYR                                                         Age in years             Numeric
         RIAGENDR                                  Sex recorded in NHANES demographics         Categorical
           BMXBMI                                                      Body mass index             Numeric
         BMXWAIST                                                  Waist circumference          

In [31]:
# Cell 31: Audit the retained DIQ010 response category

print("DIQ010 observed values after recoding:")
print(
    model_features["DIQ010"]
    .value_counts(dropna=False)
    .sort_index()
)

print("\nDIQ010 value labels from the feature documentation:")
print(
    feature_documentation.loc[
        feature_documentation["feature"] == "DIQ010"
    ].to_string(index=False)
)

print("\nResearch note:")
print(
    "DIQ010 is retained as categorical. "
    "Confirm the official response-code definitions before modelling; "
    "do not assume numeric codes represent ordered quantities."
)

DIQ010 observed values after recoding:
DIQ010
1.0    1245
2.0    6779
3.0     239
NaN       4
Name: count, dtype: int64

DIQ010 value labels from the feature documentation:
feature                                                              meaning proposed_type
 DIQ010 Ever told by a health professional that the participant has diabetes   Categorical

Research note:
DIQ010 is retained as categorical. Confirm the official response-code definitions before modelling; do not assume numeric codes represent ordered quantities.


In [32]:
# Cell 32: Define predictor types for the modelling pipeline

numeric_features = [
    "RIDAGEYR",
    "BMXBMI",
    "BMXWAIST",
    "mean_systolic_bp",
    "mean_diastolic_bp",
    "LBXGH",
]

categorical_features = [
    "RIAGENDR",
    "DIQ010",
    "HIQ011",
    "HUQ010",
    "PAQ605",
    "PAQ620",
    "PAQ650",
    "PAQ665",
    "SMQ020",
]

# Verify that every candidate predictor is assigned exactly one type
assigned_features = numeric_features + categorical_features

print("Numeric features:", len(numeric_features))
print("Categorical features:", len(categorical_features))
print("Total assigned features:", len(assigned_features))
print("Candidate predictors:", len(candidate_predictors))

print("\nFeatures missing from type assignments:")
print(sorted(set(candidate_predictors) - set(assigned_features)))

print("\nUnexpected or duplicate assignments:")
print(
    sorted(
        feature for feature in set(assigned_features)
        if assigned_features.count(feature) > 1
    )
)

print("\nObserved DIQ010 categories:")
print(
    model_features["DIQ010"]
    .value_counts(dropna=False)
    .sort_index()
)

assert set(assigned_features) == set(candidate_predictors)
assert len(assigned_features) == len(set(assigned_features))

print("\nFeature type checks passed.")

Numeric features: 6
Categorical features: 9
Total assigned features: 15
Candidate predictors: 15

Features missing from type assignments:
[]

Unexpected or duplicate assignments:
[]

Observed DIQ010 categories:
DIQ010
1.0    1245
2.0    6779
3.0     239
NaN       4
Name: count, dtype: int64

Feature type checks passed.


In [33]:
# Cell 33: Check modelling-library availability

import sklearn
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    average_precision_score,
)

print("scikit-learn version:", sklearn.__version__)

components = {
    "ColumnTransformer": ColumnTransformer,
    "Pipeline": Pipeline,
    "SimpleImputer": SimpleImputer,
    "OneHotEncoder": OneHotEncoder,
    "StandardScaler": StandardScaler,
    "StratifiedKFold": StratifiedKFold,
    "LogisticRegression": LogisticRegression,
    "RandomForestClassifier": RandomForestClassifier,
}

for name, component in components.items():
    print(f"{name}: available ({component.__name__})")

print("\nFeature configuration:")
print("Numeric:", numeric_features)
print("Categorical:", categorical_features)

scikit-learn version: 1.6.1
ColumnTransformer: available (ColumnTransformer)
Pipeline: available (Pipeline)
SimpleImputer: available (SimpleImputer)
OneHotEncoder: available (OneHotEncoder)
StandardScaler: available (StandardScaler)
StratifiedKFold: available (StratifiedKFold)
LogisticRegression: available (LogisticRegression)
RandomForestClassifier: available (RandomForestClassifier)

Feature configuration:
Numeric: ['RIDAGEYR', 'BMXBMI', 'BMXWAIST', 'mean_systolic_bp', 'mean_diastolic_bp', 'LBXGH']
Categorical: ['RIAGENDR', 'DIQ010', 'HIQ011', 'HUQ010', 'PAQ605', 'PAQ620', 'PAQ650', 'PAQ665', 'SMQ020']


In [34]:
# Cell 34: Build reusable preprocessing pipeline

numeric_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])

preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", numeric_pipeline, numeric_features),
        ("categorical", categorical_pipeline, categorical_features),
    ],
    remainder="drop",
)

print("Preprocessing pipeline created successfully.")
print("\nNumeric pipeline:")
print(numeric_pipeline)

print("\nCategorical pipeline:")
print(categorical_pipeline)

print("\nCombined preprocessor:")
print(preprocessor)

print("\nNo imputation, scaling, or encoding has been fitted yet.")
print("Original model_audit unchanged:", model_audit.shape == (8267, 22))
print("Feature dataset shape:", model_features.shape)

Preprocessing pipeline created successfully.

Numeric pipeline:
Pipeline(steps=[('imputer', SimpleImputer(strategy='median')),
                ('scaler', StandardScaler())])

Categorical pipeline:
Pipeline(steps=[('imputer', SimpleImputer(strategy='most_frequent')),
                ('onehot', OneHotEncoder(handle_unknown='ignore'))])

Combined preprocessor:
ColumnTransformer(transformers=[('numeric',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='median')),
                                                 ('scaler', StandardScaler())]),
                                 ['RIDAGEYR', 'BMXBMI', 'BMXWAIST',
                                  'mean_systolic_bp', 'mean_diastolic_bp',
                                  'LBXGH']),
                                ('categorical',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy

In [35]:
# Cell 35: Prepare X and y, then verify target balance

from sklearn.model_selection import train_test_split

X = model_features[candidate_predictors].copy()
y = model_features[target_col].astype(int).copy()

print("Feature matrix shape:", X.shape)
print("Target vector shape:", y.shape)

print("\nOverall target distribution:")
print(y.value_counts().sort_index())

print("\nOverall target percentages:")
print((y.value_counts(normalize=True).sort_index() * 100).round(2))

# Reserve 20% for a held-out test set; stratify to preserve class proportions.
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y,
)

print("\nTraining set shape:", X_train.shape)
print("Test set shape:", X_test.shape)

print("\nTraining target distribution:")
print(y_train.value_counts().sort_index())

print("\nTest target distribution:")
print(y_test.value_counts().sort_index())

print("\nTraining target percentages:")
print((y_train.value_counts(normalize=True).sort_index() * 100).round(2))

print("\nTest target percentages:")
print((y_test.value_counts(normalize=True).sort_index() * 100).round(2))

print("\nTrain/test participant overlap:", len(set(X_train.index) & set(X_test.index)))
print("Missing targets:", int(y.isna().sum()))

Feature matrix shape: (8267, 15)
Target vector shape: (8267,)

Overall target distribution:
ckd_research_indicator
0    6644
1    1623
Name: count, dtype: int64

Overall target percentages:
ckd_research_indicator
0    80.37
1    19.63
Name: proportion, dtype: float64

Training set shape: (6613, 15)
Test set shape: (1654, 15)

Training target distribution:
ckd_research_indicator
0    5315
1    1298
Name: count, dtype: int64

Test target distribution:
ckd_research_indicator
0    1329
1     325
Name: count, dtype: int64

Training target percentages:
ckd_research_indicator
0    80.37
1    19.63
Name: proportion, dtype: float64

Test target percentages:
ckd_research_indicator
0    80.35
1    19.65
Name: proportion, dtype: float64

Train/test participant overlap: 0
Missing targets: 0


In [36]:
# Cell 36: Create a logistic regression baseline pipeline

from sklearn.base import clone
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

logistic_pipeline = Pipeline(steps=[
    ("preprocessor", clone(preprocessor)),
    ("classifier", LogisticRegression(
        max_iter=2000,
        class_weight=None,
        random_state=42
    )),
])

print("Logistic regression pipeline created.")
print("\nPipeline steps:")
print(logistic_pipeline)

print("\nTraining participants:", len(X_train))
print("Held-out test participants:", len(X_test))
print("Positive cases in training:", int(y_train.sum()))
print("Positive cases in test:", int(y_test.sum()))

print("\nPipeline fitted yet:", hasattr(logistic_pipeline, "classes_"))
print("Held-out test set will remain unused during cross-validation.")

Logistic regression pipeline created.

Pipeline steps:
Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('numeric',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['RIDAGEYR', 'BMXBMI',
                                                   'BMXWAIST',
                                                   'mean_systolic_bp',
                                                   'mean_diastolic_bp',
                                                   'LBXGH']),
                                                 ('categorical',
                                                  Pipeline(steps=[('imputer',
                      

In [37]:
# Cell 37: Evaluate the logistic regression baseline with 5-fold CV

from sklearn.model_selection import StratifiedKFold, cross_validate

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

scoring = {
    "accuracy": "accuracy",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1",
    "roc_auc": "roc_auc",
    "average_precision": "average_precision",
}

cv_results = cross_validate(
    logistic_pipeline,
    X_train,
    y_train,
    cv=cv,
    scoring=scoring,
    return_train_score=False,
    n_jobs=-1,
    error_score="raise",
)

cv_summary = pd.DataFrame({
    metric: [
        cv_results[f"test_{metric}"].mean(),
        cv_results[f"test_{metric}"].std(ddof=1),
    ]
    for metric in scoring
}, index=["mean", "std"])

print("5-fold cross-validation results (training set only):")
print(cv_summary.T.round(4).to_string())

print("\nFold-level results:")
for metric in scoring:
    values = cv_results[f"test_{metric}"]
    print(f"{metric}: {[round(float(v), 4) for v in values]}")

print("\nHeld-out test set used:", False)
print("Logistic pipeline object fitted directly beforehand:",
      hasattr(logistic_pipeline, "classes_"))

5-fold cross-validation results (training set only):
                     mean     std
accuracy           0.8216  0.0099
precision          0.6305  0.0809
recall             0.2296  0.0091
f1                 0.3359  0.0190
roc_auc            0.7668  0.0211
average_precision  0.4910  0.0321

Fold-level results:
accuracy: [0.8269, 0.8095, 0.8314, 0.8124, 0.8275]
precision: [0.6824, 0.537, 0.7176, 0.5524, 0.6632]
recall: [0.2231, 0.2231, 0.2346, 0.2239, 0.2432]
f1: [0.3362, 0.3152, 0.3536, 0.3187, 0.3559]
roc_auc: [0.7942, 0.7418, 0.7734, 0.7497, 0.7751]
average_precision: [0.5237, 0.4548, 0.5113, 0.4581, 0.5071]

Held-out test set used: False
Logistic pipeline object fitted directly beforehand: False


In [38]:
# Cell 38: Evaluate class-weighted logistic regression

from sklearn.base import clone
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_validate

weighted_logistic_pipeline = Pipeline(steps=[
    ("preprocessor", clone(preprocessor)),
    ("classifier", LogisticRegression(
        max_iter=2000,
        class_weight="balanced",
        random_state=42
    )),
])

weighted_cv_results = cross_validate(
    weighted_logistic_pipeline,
    X_train,
    y_train,
    cv=cv,
    scoring=scoring,
    return_train_score=False,
    n_jobs=-1,
    error_score="raise",
)

comparison = pd.DataFrame({
    "standard_logistic_mean": [
        cv_results[f"test_{metric}"].mean() for metric in scoring
    ],
    "weighted_logistic_mean": [
        weighted_cv_results[f"test_{metric}"].mean() for metric in scoring
    ],
}, index=list(scoring.keys()))

comparison["change"] = (
    comparison["weighted_logistic_mean"]
    - comparison["standard_logistic_mean"]
)

print("Cross-validation comparison (training set only):")
print(comparison.round(4).to_string())

print("\nFold-level weighted logistic results:")
for metric in scoring:
    values = weighted_cv_results[f"test_{metric}"]
    print(f"{metric}: {[round(float(v), 4) for v in values]}")

print("\nHeld-out test set used:", False)

Cross-validation comparison (training set only):
                   standard_logistic_mean  weighted_logistic_mean  change
accuracy                           0.8216                  0.7085 -0.1131
precision                          0.6305                  0.3727 -0.2578
recall                             0.2296                  0.7064  0.4769
f1                                 0.3359                  0.4877  0.1518
roc_auc                            0.7668                  0.7659 -0.0009
average_precision                  0.4910                  0.4864 -0.0046

Fold-level weighted logistic results:
accuracy: [0.7234, 0.6803, 0.7286, 0.6952, 0.7148]
precision: [0.3957, 0.3436, 0.3922, 0.356, 0.3761]
recall: [0.7731, 0.6885, 0.6923, 0.6873, 0.6911]
f1: [0.5234, 0.4584, 0.5007, 0.469, 0.4871]
roc_auc: [0.7941, 0.7422, 0.7742, 0.7479, 0.7713]
average_precision: [0.5202, 0.4525, 0.5072, 0.4544, 0.4979]

Held-out test set used: False


In [39]:
# Cell 39: Inspect threshold trade-offs using training data only

from sklearn.base import clone
from sklearn.metrics import precision_score, recall_score, f1_score

threshold_model = clone(logistic_pipeline)
threshold_model.fit(X_train, y_train)

train_probabilities = threshold_model.predict_proba(X_train)[:, 1]

thresholds = [0.20, 0.30, 0.40, 0.50, 0.60]

threshold_rows = []

for threshold in thresholds:
    train_predictions = (train_probabilities >= threshold).astype(int)

    threshold_rows.append({
        "threshold": threshold,
        "precision": precision_score(
            y_train, train_predictions, zero_division=0
        ),
        "recall": recall_score(
            y_train, train_predictions, zero_division=0
        ),
        "f1": f1_score(
            y_train, train_predictions, zero_division=0
        ),
        "predicted_positive_count": int(train_predictions.sum()),
    })

threshold_table = pd.DataFrame(threshold_rows)

print("Training-set threshold analysis (exploratory, not unbiased evaluation):")
print(threshold_table.round(4).to_string(index=False))

print("\nHeld-out test set used:", False)
print("Threshold model fitted on training set only:", True)

Training-set threshold analysis (exploratory, not unbiased evaluation):
 threshold  precision  recall     f1  predicted_positive_count
       0.2     0.3779  0.7165 0.4948                      2461
       0.3     0.4710  0.5508 0.5078                      1518
       0.4     0.5473  0.3613 0.4353                       857
       0.5     0.6352  0.2227 0.3297                       455
       0.6     0.7273  0.1171 0.2017                       209

Held-out test set used: False
Threshold model fitted on training set only: True


In [40]:
# Cell 40: Obtain out-of-fold probabilities for threshold analysis

from sklearn.base import clone
from sklearn.model_selection import cross_val_predict

oof_probabilities = cross_val_predict(
    clone(logistic_pipeline),
    X_train,
    y_train,
    cv=cv,
    method="predict_proba",
    n_jobs=-1,
)

oof_positive_probabilities = oof_probabilities[:, 1]

oof_threshold_rows = []

for threshold in [0.20, 0.30, 0.40, 0.50, 0.60]:
    oof_predictions = (
        oof_positive_probabilities >= threshold
    ).astype(int)

    oof_threshold_rows.append({
        "threshold": threshold,
        "precision": precision_score(
            y_train, oof_predictions, zero_division=0
        ),
        "recall": recall_score(
            y_train, oof_predictions, zero_division=0
        ),
        "f1": f1_score(
            y_train, oof_predictions, zero_division=0
        ),
        "predicted_positive_count": int(oof_predictions.sum()),
    })

oof_threshold_table = pd.DataFrame(oof_threshold_rows)

print("Out-of-fold threshold analysis (training set only):")
print(oof_threshold_table.round(4).to_string(index=False))

print("\nOut-of-fold probabilities generated:", len(oof_positive_probabilities))
print("Training observations:", len(y_train))
print("Held-out test set used:", False)

Out-of-fold threshold analysis (training set only):
 threshold  precision  recall     f1  predicted_positive_count
       0.2     0.3730  0.7057 0.4880                      2456
       0.3     0.4658  0.5401 0.5002                      1505
       0.4     0.5341  0.3559 0.4272                       865
       0.5     0.6234  0.2296 0.3356                       478
       0.6     0.7143  0.1194 0.2046                       217

Out-of-fold probabilities generated: 6613
Training observations: 6613
Held-out test set used: False
